In [1]:
%%capture --no-stderr
!uv pip install langchain langchain-openai langchain-core langchain-community
!uv pip install langsmith langgraph tavily-python faiss-cpu PyMuPDF
!uv pip install python-pptx -q

In [2]:
import os
from tqdm.notebook import tqdm
from google.colab import drive, userdata
from langsmith import Client
from langchain_community.document_loaders import PyMuPDFLoader

In [4]:
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_KEY')
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = userdata.get('LANGSMITH_KEY')
os.environ["LANGCHAIN_PROJECT"] = "EAD-GenAI"

In [5]:
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
from pptx import Presentation
import os

def read_presentation_content(file_path: str) -> str:
    """
    Extracts textual content from a PowerPoint (.pptx) file.

    Args:
        file_path (str): The system path to the presentation file.

    Returns:
        str: The concatenated text found in all shapes and slides of the presentation.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"The file at {file_path} was not found.")

    if file_path.endswith('.gslides'):
        return "Error: .gslides files are Google Drive shortcuts. Please export the presentation to .pptx format to read its content."

    if not file_path.endswith('.pptx'):
        raise ValueError("Unsupported file format. Please provide a .pptx file.")

    presentation = Presentation(file_path)
    text_content = []

    for slide in presentation.slides:
        for shape in slide.shapes:
            if hasattr(shape, "text"):
                text_content.append(shape.text)

    return "\n".join(text_content)


def read_document(file_path: str) -> str:
  return PyMuPDFLoader(file_path).load()


def get_processed_data(file_path: str) -> str:
  documents = read_document(file_path)
  textual_content = ''.join([document.page_content.replace('\n', ' ') for document in documents])
  tokenizer = tiktoken.encoding_for_model("gpt-4o-mini")
  content_size = len(tokenizer.encode(textual_content))
  return textual_content, content_size

In [7]:
folder = '/content/drive/MyDrive/01 - PUC/2025-02/EAD - GenAI e Advanced Analytics/Slides/Unidade 2'

In [10]:
files_names = [file_name for file_name in os.listdir(folder) if not file_name.endswith(".gslides")]

In [11]:
files_names

['03 EAD - Ingestão de Conhecimento em LLMs (KI).pptx',
 '03 EAD - Introdução à Engenharia de Prompt.pptx',
 '03 EAD - Técnicas de Prompt.pptx',
 '04 EAD - Fine-Tuning (Dados e Pipeline).pptx',
 '05 EAD - RAG.pptx',
 '06 EAD - Personalização de Modelos.pptx']

In [12]:
 pptx_text = read_presentation_content(os.path.join(folder, files_names[0]))

In [13]:
from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model

In [14]:
content_enhancer_prompt = """
Você é um especialista em design instrucional e criação de materiais didáticos
para ensino superior. Sua tarefa é transformar conteúdo bruto extraído de
apresentações de slides em material didático estruturado, coeso e
pedagogicamente eficaz.

<contexto>
O conteúdo abaixo foi extraído automaticamente de slides de uma disciplina
universitária. A extração resulta em texto fragmentado, sem conectivos, com
redundâncias, referências visuais perdidas (como "Fonte: Elaborado pelo autor")
 e estrutura narrativa inexistente. Seu trabalho é reconstruir esse conteúdo
 como material de estudo fluido e completo.
</contexto>

<conteudo_extraido>
{slides_content}
</conteudo_extraido>

<instrucoes>
Siga estas etapas na ordem indicada:

1. ANÁLISE PRELIMINAR
   - Identifique o tema central e os subtemas abordados nos slides.
   - Mapeie a sequência lógica pretendida pelo autor original.
   - Liste conceitos-chave, definições e relações entre os tópicos.

2. REESTRUTURAÇÃO DO CONTEÚDO
   - Reorganize o material em seções com títulos descritivos.
   - Elimine redundâncias (slides frequentemente repetem títulos e tópicos).
   - Remova referências a elementos visuais que não estão disponíveis (ex:
   "conforme a figura abaixo", "Fonte: Elaborado pelo autor").
   - Mantenha as referências bibliográficas válidas, formatando-as em ABNT.

3. ENRIQUECIMENTO TEXTUAL
   - Transforme bullet points soltos em parágrafos coesos com conectivos adequados.
   - Adicione explicações intermediárias onde houver saltos lógicos entre os tópicos.
   - Inclua exemplos práticos ou analogias quando um conceito abstrato precisar
   de clarificação.
   - Preserve a precisão técnica — não simplifique a ponto de perder rigor.
</instrucoes>

<formato_de_saida>
Estruture o material assim:

# [Título do Módulo/Aula]

## [Seção 1: Título Descritivo]
Conteúdo em prosa, fluido e didático.

## [Seção 2: Título Descritivo]
Conteúdo em prosa, fluido e didático.

[... quantas seções forem necessárias ...]

## Referências
Referências bibliográficas citadas, em formato ABNT.
</formato_de_saida>

<restricoes>
- NÃO invente dados, fórmulas ou fatos que não estejam no conteúdo original.
- NÃO remova conceitos presentes no original, mesmo que pareçam secundários.
- Se houver ambiguidade no conteúdo extraído (ex: siglas não definidas, frases
 truncadas), sinalize com [VERIFICAR: descrição da dúvida] para revisão
 do professor.
- Mantenha o nível de linguagem adequado para graduação/pós-graduação.
- Escreva em português brasileiro.
</restricoes>
"""

In [15]:
large_language_model = init_chat_model("openai:gpt-5.4-nano")

In [16]:
content_enhancer_chain = (
    PromptTemplate.from_template(content_enhancer_prompt) |
    large_language_model
    )

In [17]:
%%time
response = content_enhancer_chain.invoke(
    {
        "slides_content": pptx_text
    }
)

CPU times: user 176 ms, sys: 10.2 ms, total: 186 ms
Wall time: 11.4 s


In [18]:
print(response.content)

# Ingestão de Conhecimento em LLMs (Knowledge Injection)

## 1. Ideia central: conectar um LLM genérico ao conhecimento de domínio
Modelos de linguagem (LLMs) genéricos tendem a produzir respostas a partir do conhecimento aprendido durante o treinamento em larga escala. Quando é necessário que o modelo atue com **conhecimento específico de um domínio** (por exemplo, regras de negócio, terminologia técnica, políticas internas ou dados de uma área), surge a estratégia de **Knowledge Injection** — isto é, **inserir conhecimento** no processo de geração de respostas.

A abordagem pode ser realizada por três estratégias principais:
1. **Prompt Engineering**: o conhecimento é fornecido diretamente **no contexto do prompt** apresentado ao LLM.
2. **RAG (Retrieval-Augmented Generation)**: o conhecimento é **recuperado de bases externas** durante a inferência (ou seja, no momento em que a resposta está sendo gerada).
3. **Fine-Tuning (LoRA / QLoRA)**: o conhecimento é **incorporado nos parâmetr

In [19]:
%%time
full_content = []
for file_name in tqdm(files_names):
    try:
        topic_name = file_name.replace('.pptx', '')
        full_content.append(f"\n### Tema - {topic_name}")
        pptx_text = read_presentation_content(os.path.join(folder, file_name))

        slide_content_processed = content_enhancer_chain.invoke(
            {
                "slides_content": pptx_text
            }
        ).content

        print(f"Successfully extracted {len(slide_content_processed)} characters.")
        full_content.append(slide_content_processed)
        full_content.append(180*"#")
    except Exception as e:
        print(f"Error reading file: {e}")

  0%|          | 0/6 [00:00<?, ?it/s]

Successfully extracted 4031 characters.
Successfully extracted 7547 characters.
Successfully extracted 12096 characters.
Successfully extracted 12350 characters.
Successfully extracted 5696 characters.
Successfully extracted 7059 characters.
CPU times: user 518 ms, sys: 65.9 ms, total: 584 ms
Wall time: 1min 34s


In [20]:
slides_content_string = '\n'.join(full_content)

### **Prompt de introdução a unidade**

- Registrar o prompt pois está refatorado

In [22]:
unit_introduction_prompt = """
### Papel
Você é um assistente de produção de conteúdo para disciplinas EAD da PUC Minas,
 especializado em criar materiais didáticos e pedagógicos.

### Contexto
A disciplina é **Generative AI e Advanced Analytics**. O professor fornecerá o
 conteúdo textual de uma unidade e você deverá produzir o texto de "Orientações
  de Estudo" dessa unidade.

### Conteúdo da Unidade
{unit_content}

### Instruções
Escreva o texto de Orientações de Estudo seguindo estas diretrizes:

1. Produza um texto de **2 a 3 parágrafos** (mínimo de 200 palavras).
2. Use **linguagem dialogada**, acessível e motivadora — fale diretamente com
o aluno.
3. O texto deve cobrir obrigatoriamente estes três pontos:
   - As principais temáticas e assuntos abordados na unidade.
   - Como a unidade está organizada (seções, sequência dos tópicos).
   - Os objetivos específicos de aprendizagem (o que o aluno será capaz de
   fazer ao final).

### Formato de Saída
- Texto corrido em parágrafos (sem bullet points, sem títulos internos).
- Português brasileiro, tom acolhedor e profissional.
- Não inclua saudações genéricas como "Olá, aluno!" nem despedidas como
"Bons estudos!".

### Restrições
- NÃO invente conteúdos que não estejam no material fornecido.
- NÃO use jargão excessivamente técnico sem contextualizá-lo.
- Respeite o mínimo de 200 palavras.
"""

In [23]:
unit_introduction_prompt_template = PromptTemplate.from_template(unit_introduction_prompt)

In [24]:
unit_introduction_chain = (
    unit_introduction_prompt_template |
    large_language_model
)

In [25]:
unit_introduction = unit_introduction_chain.invoke(
    {
        "unit_content": slides_content_string
    }
)

In [27]:
unit_introduction.content

'Nesta unidade, você vai entender como “injetar” conhecimento em LLMs para que eles respondam melhor em contextos de domínio (por exemplo, regras de negócio, conteúdo técnico ou padrões pedagógicos). O material começa com a ideia central de Knowledge Injection, mostrando que um modelo genérico pode não ter conhecimento específico nativo e, por isso, conectamos o LLM ao domínio. Você vê três estratégias principais: Prompt Engineering (o conhecimento vem direto no prompt), RAG (o LLM recupera trechos relevantes de uma base externa durante a inferência) e Fine-tuning com LoRA/QLoRA (o conhecimento passa a ser incorporado nos parâmetros via ajuste). Em seguida, a unidade aprofunda como escrever prompts com mais qualidade: você revisa o que é um prompt e quais elementos ajudam o modelo (instrução, contexto, dados de entrada e indicador de saída), além de delimitadores e princípios práticos para evitar ambiguidades. Depois, explora técnicas de prompting que aumentam precisão e consistência, 

### **Prompt de Conteúdo Complementar**

In [28]:
complementary_content_prompt = """
### Papel
Você é um curador de conteúdo acadêmico para disciplinas EAD da PUC Minas, especializado em selecionar materiais complementares relevantes e acessíveis para estudantes de graduação e pós-graduação.

### Contexto
A disciplina é **Generative AI e Advanced Analytics**. O professor fornecerá o conteúdo de uma unidade. Sua tarefa é recomendar materiais complementares que aprofundem os tópicos abordados.

### Conteúdo da Unidade
{unit_content}

### Instruções

#### Etapa 1 — Identificação de Tópicos
- Leia o conteúdo fornecido e identifique os 3 a 5 tópicos centrais da unidade.
- Para cada tópico, determine que tipo de material complementar seria mais útil (ex.: um artigo técnico para aprofundar teoria, um vídeo para demonstração prática, um filme para contextualização cultural).

#### Etapa 2 — Seleção de Materiais
Selecione entre **5 e 8 materiais complementares** que cubram as seguintes categorias (não é obrigatório preencher todas, mas priorize variedade):

- **Artigos acadêmicos ou técnicos** — preferencialmente open access (arXiv, Google Scholar, periódicos com acesso via biblioteca PUC Minas).
- **Vídeos** — aulas, palestras ou tutoriais disponíveis gratuitamente (YouTube, Coursera trechos abertos, TED Talks).
- **Livros ou capítulos** — disponíveis na Biblioteca Virtual da PUC Minas (Minha Biblioteca, Pearson, etc.) ou em acesso aberto.
- **Sites ou ferramentas interativas** — documentações oficiais, demos, playgrounds ou repositórios GitHub relevantes.
- **Filmes, documentários ou elementos culturais** — quando houver correlação clara com o conteúdo.

#### Etapa 3 — Contextualização
Para cada material selecionado, escreva uma contextualização de 2 a 3 frases explicando:
- O que o aluno encontrará no material.
- Como ele se conecta com o conteúdo específico da unidade.

### Formato de Saída

Para cada material, use esta estrutura:

---

**[Categoria: Artigo / Vídeo / Livro / Site / Filme]**

**Referência (ABNT):**
SOBRENOME, Nome. Título. Local: Editora, Ano. Disponível em: URL. Acesso em: dd mês. ano.

**Contextualização:**
Texto de 2-3 frases conectando o material ao conteúdo da unidade.

**Acesso:**
[Acesso gratuito / Biblioteca Virtual PUC Minas / YouTube / arXiv]

---

### Restrições
- Recomende APENAS materiais que você tenha alta confiança de que existem. Se não tiver certeza sobre a existência de um material específico, sinalize com [VERIFICAR URL] ao lado do link.
- NÃO invente títulos de artigos, livros ou vídeos. Se não conseguir lembrar o título exato, descreva o tipo de material e sugira termos de busca para o professor localizar.
- Priorize materiais em português quando disponíveis; materiais em inglês são aceitáveis, especialmente para artigos técnicos e documentações.
- Formate TODAS as referências em ABNT. Para vídeos do YouTube, use o formato de documento eletrônico.
- NÃO recomende materiais pagos sem alternativa gratuita, a menos que estejam disponíveis na Biblioteca Virtual da PUC Minas.
"""

In [29]:
complementary_content_prompt_template = (
    PromptTemplate.from_template(complementary_content_prompt)
)

In [30]:
complementary_content_chain = (
    complementary_content_prompt_template |
    large_language_model
)

In [31]:
complementary_content = complementary_content_chain.invoke(
    {
        "unit_content": slides_content_string
    }
)

In [32]:
print(complementary_content.content)

## Tópicos centrais (Etapa 1)
1. **Knowledge Injection em LLMs** (Prompt Engineering, RAG e Fine-tuning/LoRA) — *material complementar*: textos de visão geral e tutoriais práticos.  
2. **Engenharia de Prompt e Técnicas de Prompting** (CoT, Role, Self-Consistency, Reflexion, Query Expansion) — *material complementar*: guias e explicações com exemplos.  
3. **Fine-tuning: dados, formatos e pipeline (SFT, LoRA/QLoRA, templates ChatML/ChatGPT)** — *material complementar*: documentação oficial + tutoriais reprodutíveis.  
4. **RAG avançado** (Multi-Query, Busca híbrida BM25+Embeddings, RRF, Agentic RAG/Tool use) — *material complementar*: papers e guias técnicos.  
5. **Personalização de modelos** (LoRA/PEFT, RLHF vs DPO, destilação) — *material complementar*: papers seminais e guias de implementação/benchmarking.

---

## Materiais complementares (Etapa 2 e 3)

---

**[Categoria: Artigo]**

**Referência (ABNT):**  
LEWIS, Patrick et al. Retrieval-Augmented Generation for Knowledge-Intensi

In [ ]:
# print(slides_content_string)

In [36]:
unit_topics = [
    "Fine Tuning Princípios e práticas de Prompt Engineering",
    "Técnicas de Recuperação Aumentada por Geração (RAG)",
    "Estratégias de Fine-tuning",
    "Aplicações para personalização de modelos em diferentes contextos"
]

##**Prompt para topicos e objetivos da temática**

In [41]:
theme_info_prompt = """
### Papel
Você é um designer instrucional especializado em ensino superior, com experiência em estruturação de objetivos de aprendizagem pela Taxonomia de Bloom.

### Contexto
A disciplina é **Generative AI e Advanced Analytics** (EAD, PUC Minas). O professor fornecerá a temática de uma unidade e o conteúdo extraído dos slides. Sua tarefa é identificar os tópicos centrais e formular os objetivos de aprendizagem.

A disciplina tem um viés mais teorico, não terá práticas nessa abordagem

### Dados de Entrada

**Temática da Unidade:**
{unit_theme}

**Conteúdo dos Slides:**
{unit_content}

### Instruções

#### Etapa 1 — Identificação dos Tópicos Centrais
- Analise o conteúdo dos slides e extraia entre 3 e 6 tópicos centrais abordados na temática.
- Para cada tópico, escreva uma descrição de 1 a 2 frases explicando o que ele cobre.
- Ordene os tópicos na sequência lógica em que aparecem no material (do mais introdutório ao mais avançado).

#### Etapa 2 — Objetivos de Aprendizagem
- Formule entre 3 e 5 objetivos específicos de aprendizagem para a temática.
- Use como referência os **três primeiros níveis da Taxonomia de Bloom** (Lembrar, Entender, Aplicar), priorizando verbos de menor complexidade cognitiva.
- Cada objetivo deve seguir a estrutura: **verbo de ação + conteúdo + contexto/condição**.

Exemplos de verbos por nível:
- Lembrar: identificar, listar, definir, reconhecer
- Entender: descrever, explicar, diferenciar, comparar
- Aplicar: utilizar, aplicar, demonstrar, resolver

### Formato de Saída

**Tópicos Centrais:**

1. **[Nome do tópico]** — Descrição breve do que é abordado.
2. **[Nome do tópico]** — Descrição breve do que é abordado.
3. ...

**Objetivos de Aprendizagem:**

Ao final desta unidade, o aluno deverá ser capaz de:
1. [Verbo] + [o quê] + [em qual contexto].
2. [Verbo] + [o quê] + [em qual contexto].
3. ...

### Restrições
- NÃO invente tópicos ou conceitos que não estejam no conteúdo dos slides.
- NÃO utilize verbos vagos como "saber", "conhecer" ou "aprender" nos objetivos.
- NÃO ultrapasse o nível "Aplicar" da Taxonomia de Bloom, salvo se o conteúdo exigir explicitamente análise ou avaliação.
- Escreva em português brasileiro.
"""

In [42]:
theme_chain = (
    PromptTemplate.from_template(theme_info_prompt) |
    large_language_model
)

In [43]:
unit_topics[0]

'Fine Tuning Princípios e práticas de Prompt Engineering'

In [44]:
theme_response = theme_chain.invoke({
    "unit_theme": unit_topics[0],
    "unit_content": slides_content_string
})

In [45]:
print(theme_response.content)

**Tópicos Centrais:**

1. **Knowledge Injection: conectar LLM genérico ao conhecimento de domínio** — Apresenta a motivação e as formas de “injetar” conhecimento no processo de geração, distinguindo quando o conhecimento entra via prompt, via recuperação externa (RAG) ou via ajustes de parâmetros (fine-tuning).

2. **Engenharia de Prompt: estrutura e princípios para orientar a resposta** — Define prompt como instrução/contexto para guiar o LLM e detalha componentes (instrução, contexto, dados de entrada e indicador de saída), delimitadores e princípios (simplicidade, clareza e especificidade).

3. **Técnicas de Prompting: estratégias para melhorar qualidade e confiabilidade** — Explica técnicas como CoT, Role Prompting, Self Consistency, Reflexion e Query Expansion, incluindo quando usar e como a estratégia estrutura a solicitação ao modelo.

4. **Fine-tuning com LoRA/QLoRA: formatos de dados e pipeline (Unsloth)** — Descreve como preparar datasets em formatos Instruction (Alpaca) e Co

In [40]:
# client = Client()

# url = client.push_prompt(
#     prompt_identifier="unit_introduction_chain",
#     object=unit_introduction_chain,
#     tags=["EAD", "PUC"]
#     )

# print(url)

##**Geração de questões - preparação inicial**

Adicionar nos outros notebooks para criar as questões avaliativas

In [57]:
prompt_topics_from_slides = """
<role>
Você é um designer instrucional especializado em ensino superior, com
experiência em análise de conteúdo programático e estruturação de tópicos
pela Taxonomia de Bloom.
</role>

<context>
A disciplina é **Generative AI e Advanced Analytics** (EAD, PUC Minas), com
viés teórico-conceitual (sem atividades práticas/hands-on).

Sua tarefa é analisar o conteúdo extraído dos slides de uma unidade e produzir
uma lista estruturada de tópicos centrais. Esses tópicos serão usados
posteriormente como insumo para a geração de perguntas avaliativas, portanto
devem ser:
- Específicos o suficiente para permitir a formulação de questões objetivas e
dissertativas.
- Classificados pelo nível cognitivo predominante da Taxonomia de Bloom
(Lembrar, Compreender, Aplicar, Analisar, Avaliar, Criar).
- Independentes entre si, evitando sobreposições.
</context>

<task>
1. Leia atentamente o conteúdo dos slides fornecido.
2. Identifique e extraia os tópicos centrais abordados, agrupando subtópicos
relacionados quando fizer sentido.
3. Para cada tópico, forneça:
   - **Tópico**: nome claro e conciso.
   - **Descrição**: resumo em 1-2 frases do que o tópico cobre nos slides.
   - **Nível de Bloom sugerido**: o nível cognitivo mais adequado para
   avaliar esse tópico.
   - **Conceitos-chave**: lista dos termos ou conceitos essenciais associados.
4. Ordene os tópicos seguindo a sequência lógica em que aparecem no material.
</task>

<input_data>
**Conteúdo dos Slides:**
{unit_content}
</input_data>
"""

In [72]:
prompt_consolidate_topics = """
<role>
Você é um designer instrucional especializado em ensino superior, com
experiência em síntese de conteúdo programático e organização curricular.
</role>

<context>
A disciplina é **Generative AI e Advanced Analytics** (EAD, PUC Minas), com
viés teórico-conceitual (sem atividades práticas/hands-on).

Você receberá uma lista detalhada de tópicos extraídos dos slides de uma
unidade. Essa lista pode conter tópicos muito granulares ou com sobreposições
temáticas. Sua tarefa é consolidá-los em aproximadamente 15 tópicos mais
abrangentes, que serão usados como insumo para a geração de perguntas
avaliativas.
</context>

<task>
1. Analise todos os tópicos fornecidos, identificando proximidades temáticas,
   redundâncias e relações hierárquicas entre eles.
2. Agrupe tópicos relacionados sob um tópico consolidado mais abrangente.
   Tópicos que já são suficientemente distintos e relevantes devem ser
   mantidos individualmente.
3. O resultado final deve conter **aproximadamente 15 tópicos** (entre 12 e 18
   é aceitável, priorizando a cobertura completa do conteúdo).
4. Para cada tópico consolidado, forneça:
   - **Tópico**: nome claro e conciso que represente o agrupamento.
   - **Descrição**: resumo em 1-2 frases da abrangência do tópico consolidado.
   - **Nível de Bloom sugerido**: o nível cognitivo predominante para avaliar
     esse tópico (Lembrar, Compreender, Aplicar, Analisar, Avaliar, Criar).
   - **Conceitos-chave**: lista unificada dos termos e conceitos essenciais
     dos tópicos originais que foram agrupados.
5. Ordene os tópicos seguindo a sequência lógica do conteúdo da unidade.

**Critérios de agrupamento:**
- Nenhum conceito-chave dos tópicos originais deve ser descartado.
- Evite tópicos consolidados excessivamente amplos que misturem temas
  sem relação direta.
- Quando dois tópicos originais tiverem níveis de Bloom diferentes, adote
  o nível mais alto como representativo do tópico consolidado.
</task>

<input_data>
**Tópicos extraídos:**
{topics_list}
</input_data>
"""

In [58]:
from typing import List
from pydantic import BaseModel, Field

In [65]:
class SlidesTopics(BaseModel):
    topics: List[str] = Field(
        description="lista de tópicos abordados no conteúdo dos slides"
        )

class ConsolidateSlidesTopics(BaseModel):
    topics: List[str] = Field(
        description="lista resumida de tópicos abordados no conteúdo dos slides"
        )

In [73]:
slides_topics_chain = (
    PromptTemplate.from_template(prompt_topics_from_slides) |
    large_language_model.with_structured_output(SlidesTopics)
)

consolidate_slides_topics_chain = (
    PromptTemplate.from_template(prompt_consolidate_topics) |
    large_language_model.with_structured_output(ConsolidateSlidesTopics)
)

In [61]:
%%time
slides_topics = slides_topics_chain.invoke(
    {
        "unit_content": slides_content_string
    }
)

CPU times: user 53.3 ms, sys: 2.53 ms, total: 55.8 ms
Wall time: 7.49 s


In [77]:
original_topics_list

['Conceito de Knowledge Injection para conectar LLMs a conhecimento de domínio',
 'Estratégias de Knowledge Injection: Prompt Engineering, RAG e Fine-tuning (LoRA/QLoRA)',
 'Prompt Engineering: modelagem do conhecimento inserido diretamente no prompt (y=fθ(x+K))',
 'RAG (Retrieval-Augmented Generation): recuperação de contexto durante a inferência (y=fθ(x+R(q,KB)))',
 'Fine-tuning via LoRA/QLoRA: adaptação dos parâmetros do modelo (θ’=θ+Δθ)',
 'Definição de prompt e seus formatos de uso em IA (chat, imagens, programação)',
 'Interação com LLMs via chat/API e relação entre qualidade do prompt e qualidade do resultado',
 'Componentes essenciais de um prompt: instrução, contexto, dados de entrada e indicador de saída',
 'Componentes adicionais do prompt para refinar respostas: persona, critérios, restrições, formato, few-shot, detalhe, validação, idioma e estilo',
 'Delimitadores em prompts para reduzir ambiguidade (tags, Markdown e separadores)',
 'Princípios para escrever bons prompts: 

In [68]:
original_topics_list = slides_topics.topics
updated_topics_list = []

In [75]:
if len(original_topics_list) > 15:
    print("Summarizing topics list...")
    updated_slides_topics = consolidate_slides_topics_chain.invoke(
        {
            "topics_list": original_topics_list
        }
    )
    updated_topics_list = updated_slides_topics.topics

Summarizing topics list


In [76]:
updated_topics_list

['Knowledge Injection: conectar LLMs a conhecimento de domínio',
 'Estratégias de Knowledge Injection: Prompt Engineering, RAG e Fine-tuning',
 'Prompt Engineering: estrutura do conhecimento no prompt e impacto no resultado',
 'Arquitetura e componentes de prompts (instrução, contexto, entradas e saída)',
 'Técnicas avançadas de prompting para qualidade e confiabilidade (delimitadores, princípios e refinamentos)',
 'Raciocínio guiado e variações de prompting (CoT, Role Prompting, Self-Consistency, Reflexion)',
 'Expansão e reescrita de consultas para melhor recuperação (Query Expansion e critérios de escolha)',
 'Fine-tuning instrucional: necessidade de templates e formatos de dados (SFT instrucional)',
 'Fine-tuning com chat templates e formatos de conversa (ShareGPT e Llama 3.2/ChatML)',
 'Aprimoramento via LoRA/QLoRA e PEFT: parametrização e treinamento com SFTTrainer/TRL',
 'Pipeline operacional de fine-tuning e deployment/armazenamento (Unsloth, checkpoints, merge, GGUF)',
 'RAG (

##**Geração de questões**

Gerando as questões com self reflexion